In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *


In [2]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.V_45s.20241120_001500_TAI.2.Dopplergram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20250501T002002_V202602220258_0545010501.fits.gz']

In [3]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [13]:
file_hmi = files[1]
file_fdt = files[3]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data / 1000

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

data_fdt = undistort(data_fdt, header_fdt, xd, yd)
data_hmi = rebin(data_hmi, 8, update_header=header_hmi)

In [14]:
view_fdt = View.from_header(header_fdt).update(crota=header_fdt['CROTA'] + 0.29, rsun=330.25 + 0.28, xc=421.64, yc=457.87)
view_hmi = View.from_header(header_hmi)

In [18]:
data_hmi = data_hmi - view_hmi.velocity() / 1000
data_hmi -= np.nanmean(data_hmi)

plt.figure(figsize=(10,10))
plt.imshow(data_hmi, cmap='seismic', vmin=-2, vmax=2)
plt.tight_layout()

In [19]:
data_fdt = data_fdt - view_fdt.velocity() / 1000
data_fdt -= np.nanmean(data_fdt)

plt.figure(figsize=(10,10))
plt.imshow(data_fdt, cmap='seismic', vmin=-2, vmax=2)
plt.tight_layout()

In [20]:
transform = view_fdt.to_spherical(mu_thr=0.1) - view_hmi.to_spherical(mu_thr=0.1)
grid, _ = transform(view_fdt.grid)
data_hmi = bilinear(data_hmi, *grid)

In [23]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, cmap='seismic', vmin=-2, vmax=2)
plt.tight_layout()

In [24]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt, cmap='seismic', vmin=-2, vmax=2)
plt.tight_layout()

In [26]:
plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt, s=0.05)